# LingualDub — Milestone 1 & 2 Experiment Notebook (Google Colab)

**Task:** Luganda (`lug`) $\to$ English (`eng`) Speech Dubbing Baseline powered by **Sunbird AI**.

This notebook runs the full end-to-end neural speech dubbing pipeline on Colab GPU:
1. **ASR**: **Sunbird AI SALT Luganda ASR** (`Sunbird/salt-asr-luganda`)
2. **Translation**: **Sunbird AI Multilingual-to-English MT** (`Sunbird/sunbird-mul-en`)
3. **TTS**: **Meta MMS-TTS English** (`facebook/mms-tts-eng`)
4. **Evaluation**: WER, chrF, and duration error analysis
5. **Artifacts & Provenance**: Structured results saving for Git sync back to your machine.

## 1. Clone & Install LingualDub in Editable Mode

In [ ]:
# Clone repository (or pull latest if already cloned)
!git clone https://github.com/allaninfo-tech/lingualdub.git || (cd lingualdub && git pull)
%cd lingualdub

# Install LingualDub with model dependencies
!pip install -q -e ".[all]"
!pip install -q torch transformers torchaudio scipy soundfile jiwer sacrebleu

## 2. Check GPU Availability

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 3. Load Sunbird AI Baseline Pipeline from Config

In [ ]:
import lingualdub as ld
from lingualdub.cli import get_default_registry
from lingualdub.pipeline.config_loader import ConfigLoader

# 1. Initialize registry with all built-in components
registry = get_default_registry()

# 2. Load the Sunbird Luganda baseline pipeline config
loader = ConfigLoader(registry)
pipeline = loader.load_file("configs/luganda_english_baseline.yaml")

print(f"Loaded Pipeline: {pipeline.name}")
print(f"Stages: {pipeline.stage_names}")
print(f"Source Language: {pipeline.source_language} -> Target Language: {pipeline.target_language}")

## 4. Run the Dubbing Pipeline

In [ ]:
# Define sample input audio or use test clip
audio_resource = ld.Resource(
    id="luganda_sample_001",
    kind=ld.ResourceKind.SPEECH,
    language="lug",
    version="1.0.0",
    path="data/samples/sample_lug.wav",  # or provide path to your uploaded wav
    provenance={"consent_basis": "research_and_evaluation"},
)

executor = ld.PipelineExecutor(pipeline)

# Run end-to-end Sunbird ASR -> Sunbird Translation -> MMS-TTS
result = executor.run(audio_resource)
print(f"Pipeline Status: {result.status.value.upper()}")
for s in result.segments:
    print(f"[{s.start:.2f}s - {s.end:.2f}s] ({s.language}): {s.text}")

## 5. Listen to Generated Audio Artifacts

In [ ]:
from IPython.display import Audio, display

for artifact in result.artifacts:
    print(f"Playing dubbed audio: {artifact}")
    display(Audio(artifact))

## 6. Evaluate Against Reference Gold Standards

In [ ]:
from lingualdub.components.eval.metrics import WEREvaluator, TranslationEvaluator, TemporalAlignmentEvaluator

# Ground truth reference
ref_transcription = "Oli otya nnyabo, twebaza nnyo emirimu gyo."
ref_translation = "Hello madam, thank you very much for your work."

wer_eval = WEREvaluator()
trans_eval = TranslationEvaluator()
timing_eval = TemporalAlignmentEvaluator(tolerance_ms=200.0)

eval_res = wer_eval.evaluate_pair(result, ref_transcription)
eval_res = trans_eval.evaluate_pair(eval_res, ref_translation)
eval_res = timing_eval.run(eval_res)

print("Evaluation Metrics:")
import json
print(json.dumps(eval_res.metadata, indent=2))

## 7. Save Results for GitHub Sync

In [ ]:
import os
from pathlib import Path

run_dir = Path("experiments/luganda_dubbing/sunbird_colab_run_01")
run_dir.mkdir(parents=True, exist_ok=True)

# 1. Save results.json
(run_dir / "results.json").write_text(json.dumps(eval_res.to_dict(), indent=2))

# 2. Save config copy
(run_dir / "config.yaml").write_text(Path("configs/luganda_english_baseline.yaml").read_text())

print(f"Saved experiment results to {run_dir}")
print("\nTo push back to GitHub, run:")
print("!git add experiments/")
print("!git commit -m 'experiment: sunbird colab baseline run results'")
print("!git push origin main")